# Clustering Penguins: An Introduction to Unsupervised Learning

<img src = "https://github.com/jdomyancich/big-data-camp/blob/main/imgs/penguin_huddle.png?raw=true" width = 400>

In biology, classification is fundamental. Traditionally, scientists (taxonomists) would use observable traits—like the size and shape of a fish's fins or the length of a cat's tail. Today, we can collect large amounts of data and use computational tools to help us find these patterns, sometimes in ways that aren't obvious to the human eye. This is a core concept in the field of bioinformatics.

So far, we have been doing supervised learning, where the dataset is labelled. In other words, we know the right answer to the prediction. However, there are machine learning algorithms that can find patterns in unlabelled data to develop groups that can then be assigned labels. This is called **unsupervised learning**.

A great introduction to unsupervised learning is **k-means clustering**. Clustering is an unsupervised learning technique that groups similar data points together. K-means is one of the most popular clustering algorithms due to its simplicity and effectiveness.

## How to Use This Notebook

Your instructor got you started by running the first clustering pipeline with the class — exploring the data and running K-Means on two features. Now it's your turn: with your partner, work through the notebook at your own pace and, in the **Challenge**, rebuild the pipeline yourself using all four measurements.

**Start here:** run the import cell, then write the cell that loads the data — the URL is given below.

As you go:
- Wherever you see `# YOUR CODE HERE`, it's your turn to write the code.
- Check `*Hint:*` notes when you and your partner get stuck, and the **Reference Card** at the end.

## Teaching Notes: 10-Minute Launch

*(Notes for you, the instructor. Students never see this notebook.)*

**Set up two screens.** Put the **student notebook** on the projector, and keep **this instructor notebook open on a second device** as your private reference — it shows the **completed code** for every demo cell plus a short **"Building it together"** note on what to say. **You don't need to memorize anything** — read the line here and type it into the projected student notebook.

Set the scene first with the intro cell (supervised vs. unsupervised learning, and what clustering is) — just a couple of minutes.

**During the demo (~10 minutes):** type these cells live into the projected student notebook, talking through your thinking as you go:

1. `sns.pairplot(penguins)` — look for groups by eye.
2. Select two features (`bill_length_mm`, `flipper_length_mm`) and scale them with `StandardScaler`.
3. Fit `KMeans(n_clusters=3, ...)`, then pull out `.labels_` into a `predicted_cluster` column.
4. Scatter plot colored by cluster, with the centroids marked.
5. `pd.crosstab(species, predicted_cluster)` — compare the discovered clusters to the true species.

Land the point: the algorithm rediscovered most of the species groupings **without ever being told the species** — that's unsupervised learning.

**Tip:** do a quick run-through on your own before class — it makes the live typing feel easy. Note the cluster numbers (0/1/2) are arbitrary and won't line up with the species names in order.

**Then release the class.** Students redo the whole pipeline in the **Challenge** using all four measurements and check whether the match to the true species improves.

**As you circulate:** this notebook has the worked solutions (the `###` markers) — your answer key. Use the **Think About It** prompts to draw out what the clusters mean, and point fast finishers to the Challenge.

In [ ]:
# Load the necessary libraries
# data
import numpy as np
import pandas as pd

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## Load the Dataset

**→ Load `penguins.csv` into a DataFrame.**

In [ ]:
# Load the dataset from the csv file into a DataFrame called penguins, then preview it
###
penguins= pd.read_csv('https://raw.githubusercontent.com/jdomyancich/big-data-camp/refs/heads/main/data/penguins_unlabelled.csv')
###
penguins.head()

In [ ]:
# Basic statistics
penguins.describe()

### Visualizing the Data

**→ Draw `sns.pairplot(penguins)` — look for natural groups by eye.**

In [ ]:
###
sns.pairplot(penguins)
plt.show()
###

#### **Think About It:**

1. How many species are represented in the dataset? 
2. Which two features seem to be the most effective at separating the species into distinct clusters? 

## Preparing the Data for Clustering

**→ Pick two features, then scale them with `StandardScaler` (K-Means measures distance).**

In [ ]:
# Select the features for clustering
###
features = penguins[['bill_length_mm', 'flipper_length_mm']]
###

# Scale the features
###
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
###



## The K-Means Algorithm: How Does It Work?

We can see some possible clusters with our eyes, but how can a computer "discover" these clusters on their own? This is where K-Means comes in.

<img src = 'https://github.com/jdomyancich/big-data-camp/blob/main/imgs/kmeans.gif?raw=true'>

**K-Means follows a simple, iterative process:**

1.  **Choose 'k'**: First, we tell the algorithm how many clusters to look for. In our case, we know there are 3 species, so we'll choose **k=3**.
2.  **Initialize Centroids**: The algorithm randomly drops k (3) 'centroids' (think of them as virtual center-points, represented as X's above) onto the plot.
3.  **Assign Clusters**: It then assigns every single data point (each penguin) to its nearest centroid. This forms the initial clusters.
4.  **Update Centroids**: Next, it calculates the new center of each cluster by finding the average position of all the points within it. It moves the centroid to this new center.
5.  **Repeat**: It repeats steps 3 and 4 over and over. With each iteration, the centroids move less and less, until they eventually settle in the center of their respective clusters.

The algorithm has now found the groups without ever knowing the actual species!

## Applying K-Means to the Penguin Data

**→ Fit `KMeans(n_clusters=3, ...)` on the two scaled features.**

In [ ]:
# Create a KMeans model instance
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)

# Fit the model to our features
kmeans.fit(features_scaled)

print("Model training complete!")

**→ Pull the cluster labels into a new `predicted_cluster` column.**

In [ ]:
# Get the cluster assignments (labels) for each penguin
###
predicted_labels = kmeans.labels_
###

# Add the predicted labels back to our cleaned DataFrame
###
penguins['predicted_cluster'] = predicted_labels
###

# Let's look at the first few rows with the new column
penguins.head()

In [ ]:
# How many penguins are in each cluster?
###
penguins['predicted_cluster'].value_counts()
###

## Visualizing and Evaluating the Results

**→ Scatter plot colored by predicted cluster, with the centroids marked.**

In [ ]:
# Get the coordinates of the final cluster centers (the centroids)
centroids = kmeans.cluster_centers_
centroids_original_scale = scaler.inverse_transform(kmeans.cluster_centers_)

# Create a scatter plot of the penguins, colored by their assigned cluster
plt.figure(figsize=(10, 7))
sns.scatterplot(data=penguins, x='bill_length_mm', y='flipper_length_mm', hue='predicted_cluster', palette='viridis', s=100, alpha=0.7)

# Plot the centroids on top
plt.scatter(centroids_original_scale[:, 0], centroids_original_scale[:, 1], s=300, c='red', marker='X', label='Centroids')

plt.title('K-Means Clustering of Penguins')
plt.xlabel('Bill Length (mm)')
plt.ylabel('Flipper Length (mm)')
plt.legend()
plt.show()

### How good was the result?

**→ Load the labelled dataset so we can compare clusters to the true species.**

In [ ]:
# Load the labelled dataset
penguins_labelled = pd.read_csv('https://raw.githubusercontent.com/jdomyancich/big-data-camp/refs/heads/main/data/penguins_labelled.csv')
penguins_labelled['predicted_cluster'] = penguins['predicted_cluster']
penguins_labelled.head()

**→ Compare clusters to true species with `pd.crosstab` — the payoff.**

In [ ]:
###
ctab = pd.crosstab(penguins_labelled['species'], penguins_labelled['predicted_cluster'])
ctab
###

### Interpret the Results:

1.  Look at the table above. The rows are the true species, and the columns are the clusters the algorithm created. 
2.  How many 'Adelie' penguins were put into cluster 0? How many were put into other clusters?
3.  Which species was the algorithm most successful at identifying? 
4.  Where did the algorithm get confused?
5.  Overall, do you think the algorithm did a good job of rediscovering the species based only on bill and flipper length?

### Analyze the Results
- Look at the table. Does each cluster primarily correspond to a single species?
- How successful was the K-means algorithm at discovering the different penguin species without being told about them?

---

### ✋ HAND OFF TO STUDENTS — stop leading here

**You've run the full pipeline together and landed the point — the algorithm rediscovered most of the species without being told (that's unsupervised learning). Release the class now.** In the Challenge, pairs rebuild the pipeline using all four measurements. Circulate with the `###` answer key.

---

## Challenge: Can We Do Better?

What do you think would happen if we gave the algorithm more information? Let's try running K-Means again, but this time, we'll use all four numerical measurements.

*Hint: this is the same pipeline you just ran, with more columns. Select all four measurements (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`), scale them with `StandardScaler`, fit a new `KMeans(n_clusters=3, n_init=10, random_state=42)`, and store the labels in a new column such as `predicted_cluster_all`.*

In [ ]:
# Build the clustering pipeline again using all four measurements
###
features_all = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
scaler_all = StandardScaler()
features_all_scaled = scaler_all.fit_transform(features_all)

kmeans_all = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans_all.fit(features_all_scaled)

penguins['predicted_cluster_all'] = kmeans_all.labels_
penguins.head()
###

*Hint: add your new cluster labels to `penguins_labelled`, then build a `pd.crosstab` of the true `species` against those labels — just like before — to see if using all four features improved the match.*

In [ ]:
# Compare the new clusters to the true species with a crosstab
###
penguins_labelled['predicted_cluster_all'] = penguins['predicted_cluster_all']
pd.crosstab(penguins_labelled['species'], penguins_labelled['predicted_cluster_all'])
###

---

### 👥 REGROUP THE CLASS — come back together here

**Once pairs have run the four-feature version, reconvene** and compare its crosstab to the two-feature result — did more features produce a cleaner match to the true species?

---

#### Think About It

1. Did using all four measurements produce a cleaner match to the true species than two features did?
2. Why might giving the algorithm more features help it separate the groups?

## Reference Card

Stuck? Here are the key patterns from today, all in one place.

**Explore feature pairs visually:**
`sns.pairplot(penguins)`

**Select features and scale them** (K-Means measures distance, so scaling matters):
`features = penguins[['bill_length_mm', 'flipper_length_mm']]`
`scaler = StandardScaler()`
`features_scaled = scaler.fit_transform(features)`

**Create and fit a K-Means model:**
`kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)`
`kmeans.fit(features_scaled)`

**Get the cluster labels and add them as a column:**
`penguins['predicted_cluster'] = kmeans.labels_`

**Count how many points landed in each cluster:**
`penguins['predicted_cluster'].value_counts()`

**Compare clusters to the true labels:**
`pd.crosstab(penguins_labelled['species'], penguins_labelled['predicted_cluster'])`

Because clustering is *unsupervised*, the cluster numbers (0, 1, 2) are arbitrary — cluster 0 won't necessarily be 'Adelie'.